# 00. PyTorch Fundamentals

This notebook covers the foundational building block of PyTorch: the **tensor**.

It's based on the excellent [Zero to Mastery — Learn PyTorch for Deep Learning](https://www.learnpytorch.io/00_pytorch_fundamentals/) course by Daniel Bourke, expanded here with extra explanations, missing sections filled in, and a couple of bugs fixed.

**What we'll cover:**

| Topic | What it's about |
|---|---|
| Introduction to tensors | Tensors are the basic building block of all ML/DL |
| Creating tensors | Tensors can represent almost any kind of data |
| Getting information from tensors | Reading shape, dtype and device off a tensor |
| Manipulating tensors | Adding, multiplying, matrix-multiplying tensors |
| Dealing with tensor shapes | Avoiding (and fixing) shape-mismatch errors |
| Reshaping / stacking / squeezing | Changing a tensor's shape without changing its data |
| Indexing on tensors | Selecting data out of a tensor |
| Mixing PyTorch tensors and NumPy | Converting between `torch.Tensor` and `np.ndarray` |
| Reproducibility | Taming randomness with random seeds |
| Running tensors on the GPU | Making tensor operations much faster |

> 💡 **Tip:** whenever you get stuck, the [PyTorch documentation](https://pytorch.org/docs/stable/index.html) and the [PyTorch forums](https://discuss.pytorch.org/) are your best friends.

## 0. Importing PyTorch and checking our setup

Before anything else, let's import PyTorch and check which version we have and whether a GPU is available.

> **Note:** if you're running this in Google Colab, PyTorch is already installed. If you're running locally, follow the [official install guide](https://pytorch.org/get-started/locally/) first.

In [1]:
import torch
print(torch.__version__)              # shows installed PyTorch version
print(torch.cuda.is_available())      # True if a CUDA-capable GPU is available

2.13.0+cu130
False


If a GPU is available, `!nvidia-smi` (a command line tool from NVIDIA) prints out details about it — driver version, CUDA version, memory usage, etc. If you don't have an NVIDIA GPU, this cell will print an error, which is fine — the rest of this notebook works perfectly well on CPU.

In [2]:
!nvidia-smi   # About GPU

/usr/bin/sh: 1: nvidia-smi: not found


Now let's import the other libraries we'll use throughout this notebook.

In [3]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Introduction to tensors

Tensors are the fundamental building block of machine learning. Their job is to represent data **numerically**.

For example, an image could be represented as a tensor with shape `[3, 224, 224]`, meaning `[colour_channels, height, width]` — 3 colour channels (red, green, blue), a height of 224 pixels, and a width of 224 pixels.

In "tensor-speak", a tensor's **dimensionality** (`ndim`) is how many indices you need to reach a single value inside it. Let's build up from the simplest possible tensor (a single number) to more complex ones.

| Name | What is it? | Number of dimensions | Typical notation |
|---|---|---|---|
| **scalar** | a single number | 0 | lowercase, e.g. `a` |
| **vector** | a number with direction (can hold many numbers) | 1 | lowercase, e.g. `y` |
| **matrix** | a 2-dimensional array of numbers | 2 | uppercase, e.g. `Q` |
| **tensor** | an n-dimensional array of numbers | any number | uppercase, e.g. `X` |

> 💡 **Trick:** you can count a tensor's dimensions by counting the number of square brackets on one side of the printed tensor.

### Creating tensors

#### Scalar
A scalar is a single number — a zero-dimensional tensor.

In [4]:
# Scalar
scalar = torch.tensor(7)
print(scalar)
print(scalar.ndim)
print(scalar.item())   # .item() pulls the plain Python number out of a one-element tensor

tensor(7)
0
7


#### Vector
A vector is a single-dimension tensor, but it can hold many numbers — e.g. `[bedrooms, bathrooms]` for a house, or `[bedrooms, bathrooms, car_parks]`.

In [5]:
# Vector
vector = torch.tensor([7, 7])
print(vector)
print(vector.ndim)
print(vector.shape)

tensor([7, 7])
1
torch.Size([2])


#### Matrix
Matrices have an extra dimension compared to vectors.

In [6]:
# Matrix
MATRIX = torch.tensor([[7, 8],
                       [9, 10],
                       [11, 23]])
print(MATRIX)
print(MATRIX.shape)
print(MATRIX.ndim)

tensor([[ 7,  8],
        [ 9, 10],
        [11, 23]])
torch.Size([3, 2])
2


#### Tensor
A tensor can have any number of dimensions. Here's a 3-dimensional one — think of it as a stack of matrices.

In [7]:
# Tensor
TENSOR = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
print(TENSOR)
print(TENSOR.ndim)
print(TENSOR.shape)

tensor([[[1, 2, 3],
         [3, 6, 9],
         [2, 4, 5]]])
3
torch.Size([1, 3, 3])


> **Note:** by convention, scalars and vectors are named with lowercase variables (`scalar`, `vector`) while matrices and tensors get uppercase names (`MATRIX`, `TENSOR`). You'll also see people use "matrix" and "tensor" interchangeably — in PyTorch, everything is technically a `torch.Tensor`, and its shape decides what it actually represents.

### Random tensors

Why random tensors? Neural networks rarely start out with hand-crafted tensors like the ones above. Instead, they start with tensors full of random numbers and adjust those numbers as they learn patterns in data:

`Start with random numbers -> look at data -> update numbers -> look at data -> update numbers -> ...`

We create random tensors with `torch.rand(size=...)`.

In [8]:
# Create a random tensor of size (3, 4)
random_tensor = torch.rand(size=(3, 4))   # (height, width) style shape
print(random_tensor)
print(random_tensor.dtype)
print(random_tensor.shape)
print(random_tensor.ndim)

tensor([[0.3554, 0.0868, 0.7668, 0.9941],
        [0.8915, 0.7373, 0.7698, 0.2578],
        [0.4289, 0.0167, 0.1377, 0.1881]])
torch.float32
torch.Size([3, 4])
2


In [9]:
# Create a random tensor in a common image shape: [height, width, colour_channels]
random_image_size_tensor = torch.rand(size=(224, 224, 3))
print(random_image_size_tensor.shape)
print(random_image_size_tensor.ndim)

torch.Size([224, 224, 3])
3


### Zeros and ones

Sometimes you'll want tensors that are entirely `0`s or `1`s — this is common for **masking** values you don't want a model to learn from.

In [10]:
# Creating tensors with zeros and ones
zeros = torch.zeros(size=(3, 4))
print(zeros)
print(zeros.dtype)

ones = torch.ones(size=(3, 4))
print(ones)
print(ones.dtype)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
torch.float32
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
torch.float32


### Creating a range and tensors_like

Use `torch.arange(start, end, step)` for a range of values. `torch.range()` still runs in most versions but is **deprecated** — always prefer `torch.arange()`.

In [11]:
# Use torch.arange(), torch.range() is deprecated
zero_to_ten_deprecated = torch.range(0, 10)  # Note: this may error in future PyTorch versions

# Create a range of values 0 to 10
zero_to_ten = torch.arange(start=0, end=10, step=1)
zero_to_ten

/tmp/ipykernel_644/2079374839.py:2: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  zero_to_ten_deprecated = torch.range(0, 10)  # Note: this may error in future PyTorch versions


tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

`torch.zeros_like(input)` / `torch.ones_like(input)` create a tensor of zeros/ones with the **same shape** as another tensor — handy when you don't want to type the shape out by hand.

In [12]:
# Create a tensor of zeros with the same shape as zero_to_ten
ten_zeros = torch.zeros_like(input=zero_to_ten)
ten_zeros

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

### Tensor datatypes

Tensor datatypes are one of the [3 big errors you'll run into with PyTorch](https://www.learnpytorch.io/pytorch_most_common_errors/) (the other two being shape and device errors).

Some datatypes are built for CPUs, some for GPUs. The most common (and the default) is `torch.float32` — "32-bit floating point". You'll also see `torch.float16` (half precision) and `torch.float64` (double precision), plus various integer types (`torch.int8`, `torch.int32`, `torch.int64`, etc).

**Why does precision matter?** The higher the number of bits, the more detail a number is stored with — but also the more memory and compute it costs. Lower precision (`float16`, `int8`) is faster and smaller but less accurate; this trade-off matters a lot when doing millions of operations, or deploying a model to a phone.

Three tensor attributes worth knowing cold:
- `dtype` — what datatype are the elements stored as?
- `device` — is the tensor on the `cpu` or `cuda` (GPU)?
- `requires_grad` — should PyTorch track gradients for this tensor? (used heavily once we start training models)

In [13]:
# Default datatype for tensors is float32
float_32_tensor = torch.tensor([3.0, 6.0, 9.0],
                                dtype=None,           # defaults to None -> torch.float32
                                device=None,          # defaults to None -> uses the default tensor type (usually CPU)
                                requires_grad=False)  # if True, PyTorch tracks operations on this tensor for autograd

print(float_32_tensor.shape, float_32_tensor.dtype, float_32_tensor.device)

torch.Size([3]) torch.float32 cpu


In [14]:
# Create the same values but store them as float16 instead
float_16_tensor = torch.tensor([3.0, 6.0, 9.0], dtype=torch.float16)  # torch.half also works
print(float_16_tensor.dtype)

torch.float16


> **Rule of thumb:** most PyTorch errors trace back to a mismatch between two tensors' *shape*, *dtype*, or *device*. When something breaks, ask yourself: "what shape, what dtype, where (which device)?"

## 2. Getting information from tensors

Once you (or a PyTorch layer) create a tensor, you'll often want to inspect it. The three attributes to check first:
- `tensor.shape` — the size of each dimension
- `tensor.dtype` — the datatype of its elements
- `tensor.device` — `cpu` or `cuda`

In [15]:
some_tensor = torch.rand(3, 4)

print(some_tensor)
print(f"Shape of tensor: {some_tensor.shape}")
print(f"Datatype of tensor: {some_tensor.dtype}")
print(f"Device tensor is stored on: {some_tensor.device}")   # defaults to cpu

tensor([[0.6132, 0.4856, 0.0669, 0.5779],
        [0.7967, 0.9562, 0.3187, 0.2153],
        [0.0266, 0.4710, 0.2330, 0.7459]])
Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


## 3. Manipulating tensors (tensor operations)

A neural network learns by performing huge numbers of tensor operations, end-to-end. Almost all of them boil down to just a handful of building blocks:
- Addition
- Subtraction
- Multiplication (element-wise)
- Division
- Matrix multiplication

### Basic operations

The standard Python operators (`+ - * /`) work exactly how you'd expect on tensors, applying the operation **element-wise**.

In [16]:
tensor = torch.tensor([1, 2, 3])

print(tensor + 10)   # add 10 to every element
print(tensor * 10)   # multiply every element by 10
print(tensor - 10)   # subtract 10 from every element
print(tensor / 10)   # divide every element by 10

tensor([11, 12, 13])
tensor([10, 20, 30])
tensor([-9, -8, -7])
tensor([0.1000, 0.2000, 0.3000])


These operations **don't** change `tensor` unless you reassign it — PyTorch tensor operations are, by default, out-of-place (a new tensor is returned).

In [17]:
# tensor is unchanged unless we reassign it
print(tensor)

tensor = tensor + 10   # now it's reassigned
print(tensor)

tensor([1, 2, 3])
tensor([11, 12, 13])


PyTorch also has function equivalents like `torch.add()` and `torch.mul()`, but using the operators (`+ - * /`) is more common and idiomatic.

In [18]:
print(torch.multiply(tensor, 10))
print(torch.add(tensor, 10))

# Element-wise multiplication (each element multiplies with its own equivalent: index 0*0, 1*1, 2*2 ...)
print(tensor, "*", tensor)
print("Equals:", tensor * tensor)

tensor([110, 120, 130])
tensor([21, 22, 23])
tensor([11, 12, 13]) * tensor([11, 12, 13])
Equals: tensor([121, 144, 169])


### Matrix multiplication (the most important operation in deep learning)

Matrix multiplication sits at the heart of almost every neural network layer. PyTorch implements it with `torch.matmul()` (aliases: `torch.mm()`, and the `@` operator).

**Two rules to memorise:**

1. The **inner dimensions** must match:
   - `(3, 2) @ (3, 2)` ❌ won't work
   - `(2, 3) @ (3, 2)` ✅ works
   - `(3, 2) @ (2, 3)` ✅ works
2. The result takes the shape of the **outer dimensions**:
   - `(2, 3) @ (3, 2)` → `(2, 2)`
   - `(3, 2) @ (2, 3)` → `(3, 3)`

The difference between element-wise multiplication and matrix multiplication is that matrix multiplication **sums** the products:

| Operation | Calculation | Code |
|---|---|---|
| Element-wise | `[1*1, 2*2, 3*3]` = `[1, 4, 9]` | `tensor * tensor` |
| Matrix multiplication | `1*1 + 2*2 + 3*3` = `14` | `torch.matmul(tensor, tensor)` |

In [19]:
tensor = torch.tensor([1, 2, 3])

# Element-wise multiplication
print(tensor * tensor)

# Matrix multiplication (a dot product, since these are 1D tensors)
print(torch.matmul(tensor, tensor))

# The "@" symbol also works but is less explicit/readable - matmul() is preferred
print(tensor @ tensor)

tensor([1, 4, 9])
tensor(14)
tensor(14)


`torch.matmul()` is implemented in optimised C/CUDA code and will always be dramatically faster than looping over elements by hand in Python — avoid manual for-loops over tensors wherever possible.

In [20]:
%%time
# Matrix multiplication "by hand" with a Python loop (slow - don't do this in practice)
value = 0
for i in range(len(tensor)):
    value += tensor[i] * tensor[i]
print(value)

tensor(14)
CPU times: user 1.29 ms, sys: 292 μs, total: 1.58 ms
Wall time: 2.53 ms


In [21]:
%%time
# The built-in, vectorised version - much faster
print(torch.matmul(tensor, tensor))

tensor(14)
CPU times: user 594 μs, sys: 134 μs, total: 728 μs
Wall time: 1.43 ms


### One of the most common errors in deep learning: shape mismatches

Because matrix multiplication has strict rules about which shapes can be combined, shape errors are one of the most common bugs you'll run into. Let's deliberately trigger one, then fix it.

In [22]:
tensor_A = torch.tensor([[1, 2],
                         [3, 4],
                         [5, 6]], dtype=torch.float32)

tensor_B = torch.tensor([[7, 10],
                         [8, 11],
                         [9, 12]], dtype=torch.float32)

# Both tensors are (3, 2) - inner dimensions (2 and 3) don't match, so this will error
try:
    torch.matmul(tensor_A, tensor_B)
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)


To fix it, we transpose one of the tensors so the inner dimensions line up. `tensor.T` (or `torch.transpose(input, dim0, dim1)`) swaps a tensor's dimensions.

In [23]:
tensor_c = tensor_B.T
print(f"tensor_A shape:   {tensor_A.shape}")
print(f"tensor_B.T shape: {tensor_c.shape}  <- inner dimensions now match (2 and 2)")

output = torch.matmul(tensor_A, tensor_c)
print(output)
print(f"Output shape: {output.shape}")

tensor_A shape:   torch.Size([3, 2])
tensor_B.T shape: torch.Size([2, 3])  <- inner dimensions now match (2 and 2)
tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]])
Output shape: torch.Size([3, 3])


> **Sneak peek:** neural network layers like `torch.nn.Linear()` are just matrix multiplication (`y = x · Aᵀ + b`) under the hood, where `A` is a weights matrix that starts random and gets updated as the model learns. We'll use this a lot in later notebooks.

In [24]:
torch.manual_seed(42)
linear = torch.nn.Linear(in_features=2, out_features=6)   # in_features must match the inner dimension of the input
x = tensor_A
output = linear(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([3, 2])
Output shape: torch.Size([3, 6])


### Finding the min, max, mean and sum (aggregation)

"Aggregation" operations take many values and reduce them down to fewer — often just one.

In [25]:
x = torch.rand(size=(3, 2))
print(x)
print("shape =", x.shape)
print("max   =", x.max())
print("min   =", x.min())
print("mean  =", x.mean())   # NOTE: .mean() requires a float dtype
print("sum   =", x.sum())

tensor([[0.2666, 0.6274],
        [0.2696, 0.4414],
        [0.2969, 0.8317]])
shape = torch.Size([3, 2])
max   = tensor(0.8317)
min   = tensor(0.2666)
mean  = tensor(0.4556)
sum   = tensor(2.7336)


> **Note:** `.mean()` requires a floating-point dtype. If you call it on an integer tensor, cast first: `tensor.type(torch.float32).mean()`.

### Positional min/max

`torch.argmax()` / `torch.argmin()` return the **index** where the max/min value occurs, rather than the value itself. This is useful when you only care about *where* the highest value is (we'll see this again with the softmax activation function later on).

In [26]:
# Create a tensor
tensor = torch.arange(10, 100, 10)
print(f"Tensor: {tensor}")

# Returns index of max and min values
print(f"Index where max value occurs: {tensor.argmax()}")
print(f"Index where min value occurs: {tensor.argmin()}")

Tensor: tensor([10, 20, 30, 40, 50, 60, 70, 80, 90])
Index where max value occurs: 8
Index where min value occurs: 0


### Change tensor datatype

You'll often need to cast a tensor from one dtype to another — for example to satisfy an operation like `.mean()`, or to save memory. Use `tensor.type(dtype)`.

In [27]:
tensor = torch.arange(10., 100., 10.)
print(tensor.dtype)   # float32 by default here since we used floats in arange

tensor_float16 = tensor.type(torch.float16)
print(tensor_float16)

tensor_int8 = tensor.type(torch.int8)
print(tensor_int8)

torch.float32


tensor([10., 20., 30., 40., 50., 60., 70., 80., 90.], dtype=torch.float16)
tensor([10, 20, 30, 40, 50, 60, 70, 80, 90], dtype=torch.int8)


## 4. Reshaping, stacking, squeezing and unsqueezing

Often you want to change a tensor's shape *without* changing the values inside it. The common tools are:

| Method | What it does |
|---|---|
| `torch.reshape(input, shape)` / `tensor.reshape(shape)` | Reshapes a tensor to `shape` (if the number of elements is compatible) |
| `tensor.view(shape)` | Returns a **view** of the tensor in a new shape — shares the same underlying data |
| `torch.stack(tensors, dim=0)` | Concatenates a sequence of tensors along a **new** dimension |
| `torch.cat(tensors, dim=0)` | Concatenates tensors along an **existing** dimension (no new dim added) |
| `torch.squeeze(input)` | Removes all dimensions of size `1` |
| `torch.unsqueeze(input, dim)` | Adds a dimension of size `1` at position `dim` |
| `torch.permute(input, dims)` | Returns a **view** with dimensions rearranged (reordered) to `dims` |

Let's go through these one at a time, starting with reshape.

In [28]:
# Create a tensor
import torch
x = torch.rand(size=(2, 3))
x_reshaped = x.reshape(3, 2)
print(x)

print("   ")
print()
print(x_reshaped)
print(x_reshaped.shape)

tensor([[0.1053, 0.2695, 0.3588],
        [0.1994, 0.5472, 0.0062]])
   

tensor([[0.1053, 0.2695],
        [0.3588, 0.1994],
        [0.5472, 0.0062]])
torch.Size([3, 2])


In [29]:
# changing the view
# Create a tensor
import torch
x = torch.rand(size=(2, 3))

# Change view (keeps same data as original but changes view)
# See more: https://stackoverflow.com/a/54507446/7900723
z = x.view(1, 6)
t = x.T
print(x)
print(x.shape)
print(z)
print(z.shape)
print(t.reshape(1, 6))
print(t.shape)

tensor([[0.9516, 0.0753, 0.8860],
        [0.5832, 0.3376, 0.8090]])
torch.Size([2, 3])
tensor([[0.9516, 0.0753, 0.8860, 0.5832, 0.3376, 0.8090]])
torch.Size([1, 6])
tensor([[0.9516, 0.5832, 0.0753, 0.3376, 0.8860, 0.8090]])
torch.Size([3, 2])


1. **`.view()`**

**Reshapes a tensor by reinterpreting its memory layout.**

Requires contiguous memory (elements stored sequentially).

Fast, but will throw an error if the tensor is non-contiguous (like after a transpose).

Does not copy data — just changes how it's "viewed."

2. **`.reshape()`**

Also reshapes a tensor, but more flexible.

Works on both contiguous and non-contiguous tensors.

If needed, it will make a copy of the data to achieve the new shape.

Safer than `.view()` when you're unsure about contiguity.

3. **`.transpose()`** (or `.T`)

Swaps dimensions (rows ↔ columns for 2D).

Changes the shape but not the number of elements.

**Often results in a non-contiguous tensor because it changes how strides access memory.**

Example: `(2, 3) → (3, 2)`.

In [30]:
# Because view() shares the same underlying data as the original tensor,
# changing the view changes the original tensor too!
x = torch.arange(1., 8.)
z = x.view(1, 7)
z[:, 0] = 5
print(z)
print(x)   # x[0] also became 5 - x and z share memory

tensor([[5., 2., 3., 4., 5., 6., 7.]])
tensor([5., 2., 3., 4., 5., 6., 7.])


### Stacking tensors

`torch.stack()` combines a list of tensors along a **new** dimension — all the tensors must be the same shape.

In [31]:
x = torch.arange(1., 8.)

# Stack x on top of itself 4 times along a new dimension 0
x_stacked = torch.stack([x, x, x, x], dim=0)
print(x_stacked)
print(x_stacked.shape)

# Try dim=1 to see the difference
x_stacked_dim1 = torch.stack([x, x, x, x], dim=1)
print(x_stacked_dim1.shape)

tensor([[1., 2., 3., 4., 5., 6., 7.],
        [1., 2., 3., 4., 5., 6., 7.],
        [1., 2., 3., 4., 5., 6., 7.],
        [1., 2., 3., 4., 5., 6., 7.]])
torch.Size([4, 7])
torch.Size([7, 4])


### Squeezing a tensor

`torch.squeeze()` removes all dimensions of size `1` from a tensor.

In [32]:
x_reshaped = x.reshape(1, 7)
print(f"Previous tensor: {x_reshaped}")
print(f"Previous shape: {x_reshaped.shape}")

x_squeezed = x_reshaped.squeeze()
print(f"\nNew tensor: {x_squeezed}")
print(f"New shape: {x_squeezed.shape}")

Previous tensor: tensor([[1., 2., 3., 4., 5., 6., 7.]])
Previous shape: torch.Size([1, 7])

New tensor: tensor([1., 2., 3., 4., 5., 6., 7.])
New shape: torch.Size([7])


### Unsqueezing a tensor

`torch.unsqueeze()` does the reverse of `squeeze()` — it adds a dimension of size `1` back in at a chosen index (`dim`).

In [33]:
print(f"Previous tensor: {x_squeezed}")
print(f"Previous shape: {x_squeezed.shape}")

x_unsqueezed = x_squeezed.unsqueeze(dim=0)
print(f"\nNew tensor: {x_unsqueezed}")
print(f"New shape: {x_unsqueezed.shape}")

Previous tensor: tensor([1., 2., 3., 4., 5., 6., 7.])
Previous shape: torch.Size([7])

New tensor: tensor([[1., 2., 3., 4., 5., 6., 7.]])
New shape: torch.Size([1, 7])


### Permuting a tensor

`torch.permute()` rearranges (reorders) a tensor's dimensions. This is common with image tensors, where you might need to switch between `[height, width, colour_channels]` and `[colour_channels, height, width]`.

In [34]:
x_original = torch.rand(size=(224, 224, 3))   # [height, width, colour_channels]

# Rearrange axis order: 0->1, 1->2, 2->0
x_permuted = x_original.permute(2, 0, 1)      # now [colour_channels, height, width]

print(f"Previous shape: {x_original.shape}")
print(f"New shape: {x_permuted.shape}")

Previous shape: torch.Size([224, 224, 3])
New shape: torch.Size([3, 224, 224])


> **Note:** just like `.view()`, `.permute()` returns a **view** — it shares the same underlying data as the original tensor, so changing values in the permuted tensor changes the original too.

## 5. Indexing (selecting data from tensors)

Indexing into a PyTorch tensor works a lot like indexing into a Python list or a NumPy array — you just have more dimensions to work with. Indexing moves from the **outer** dimension to the **inner** dimension.

In [35]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x)
print(x.shape)

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])
torch.Size([1, 3, 3])


In [36]:
# Index bracket by bracket, outer -> inner
print(f"First square bracket:\n{x[0]}")
print(f"Second square bracket: {x[0][0]}")
print(f"Third square bracket: {x[0][0][0]}")

First square bracket:
tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
Second square bracket: tensor([1, 2, 3])
Third square bracket: 1


You can also use `:` to mean "every value along this dimension".

In [37]:
# All values of dim 0, index 0 of dim 1
print(x[:, 0])

# All values of dims 0 & 1, only index 1 of dim 2
print(x[:, :, 1])

# All of dim 0, only index 1 of dims 1 & 2
print(x[:, 1, 1])

# Index 0 of dims 0 & 1, all of dim 2 (same as x[0][0])
print(x[0, 0, :])

tensor([[1, 2, 3]])
tensor([[2, 5, 8]])
tensor([5])
tensor([1, 2, 3])


## 6. PyTorch tensors & NumPy

PyTorch plays nicely with NumPy. The two methods you'll use most:
- `torch.from_numpy(ndarray)` → NumPy array to PyTorch tensor
- `tensor.numpy()` → PyTorch tensor to NumPy array

> **Note:** NumPy's default dtype is `float64`, while PyTorch's default is `float32`. Converting a NumPy array to a tensor keeps `float64` unless you explicitly cast it: `torch.from_numpy(array).type(torch.float32)`.

NumPy tends to be very fast on CPU, but PyTorch tensors are typically much faster than NumPy once you move them onto a GPU.

In [38]:
# NumPy array to tensor
import torch
import numpy as np
array = np.arange(1.0, 8.0)
tensor = torch.from_numpy(array)
print(array, tensor)
print(tensor.dtype)   # float64 - inherited from the NumPy array

[1. 2. 3. 4. 5. 6. 7.] tensor([1., 2., 3., 4., 5., 6., 7.], dtype=torch.float64)
torch.float64


> **Careful:** `torch.from_numpy()` actually shares the same underlying memory as the original NumPy array. Below, `array = array + 1` creates a **brand new** array object (reassignment), so `tensor` is left pointing at the old memory and doesn't change. But an in-place op like `array += 1` would modify the shared memory, and `tensor` would change too. When in doubt, use `tensor.clone()` or `array.copy()` to break the link.

In [39]:
# Change the array, keep the tensor (because we reassigned `array` above, not modified in place)
array = array + 1
print(array, tensor)

[2. 3. 4. 5. 6. 7. 8.] 

tensor([1., 2., 3., 4., 5., 6., 7.], dtype=torch.float64)


In [40]:
# converting pytorch to numpy
import torch
import numpy as np

tensors = torch.rand(size=(1, 4))
array_1 = torch.Tensor.numpy(tensors)
print(tensors)
print(array_1)

tensor([[0.1336, 0.9666, 0.9754, 0.8474]])
[[0.13360566 0.96661294 0.9753964  0.847449  ]]


## 7. Reproducibility (taking the random out of random)

PyTorch's randomness is actually **pseudo**-randomness — deterministic under the hood, but designed to look random. Setting a **random seed** with `torch.manual_seed(seed)` "flavours" that randomness so you get the same numbers every time, which is essential for reproducible experiments.

In [41]:
import torch

# Create two random tensors - since there's no seed set, we'd expect them to differ
random_tensor_A = torch.rand(3, 4)
random_tensor_B = torch.rand(3, 4)

print(f"Tensor A:\n{random_tensor_A}\n")
print(f"Tensor B:\n{random_tensor_B}\n")
print("Does Tensor A equal Tensor B? (anywhere)")
print(random_tensor_A == random_tensor_B)

Tensor A:
tensor([[0.8988, 0.1105, 0.4563, 0.9719],
        [0.3968, 0.1496, 0.4743, 0.9973],
        [0.4436, 0.9726, 0.5194, 0.5337]])

Tensor B:
tensor([[0.7050, 0.3362, 0.7891, 0.1694],
        [0.1800, 0.7177, 0.6988, 0.5510],
        [0.2485, 0.8518, 0.0963, 0.1338]])

Does Tensor A equal Tensor B? (anywhere)
tensor([[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]])


In [42]:
import torch
import random

# # Set the random seed
RANDOM_SEED = 0   # try changing this to different values and see what happens to the numbers below
torch.manual_seed(seed=RANDOM_SEED)
random_tensor_C = torch.rand(3, 4)

# Have to reset the seed every time a new rand() is called
# Without this, tensor_D would be different to tensor_C
torch.random.manual_seed(seed=RANDOM_SEED)   # try commenting this line out and seeing what happens
random_tensor_D = torch.rand(3, 4)

print(f"Tensor C:\n{random_tensor_C}\n")
print(f"Tensor D:\n{random_tensor_D}\n")
print("Does Tensor C equal Tensor D? (anywhere)")
random_tensor_C == random_tensor_D

Tensor C:
tensor([[0.4963, 0.7682, 0.0885, 0.1320],
        [0.3074, 0.6341, 0.4901, 0.8964],
        [0.4556, 0.6323, 0.3489, 0.4017]])

Tensor D:
tensor([[0.4963, 0.7682, 0.0885, 0.1320],
        [0.3074, 0.6341, 0.4901, 0.8964],
        [0.4556, 0.6323, 0.3489, 0.4017]])

Does Tensor C equal Tensor D? (anywhere)


tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])

## 8. Running tensors on GPUs (and making things faster)

Deep learning involves huge numbers of numerical operations, and GPUs are typically much faster than CPUs at exactly the kind of matrix multiplication tensors need. PyTorch makes it straightforward to move data (and models) onto a GPU when one's available.

> **Note:** "GPU" here means an NVIDIA GPU with CUDA enabled, unless stated otherwise. If you're on Apple Silicon (M1/M2/M3), see the `torch.backends.mps` note below.

In [43]:
# Check for GPU
import torch
torch.cuda.is_available()

False

### Writing device-agnostic code

It's best practice to write code that works whether or not a GPU is available — this is called **device-agnostic code**. A common pattern:

In [44]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

> **Apple Silicon (M1/M2/M3):** use `torch.backends.mps.is_available()` instead of (or alongside) the CUDA check:
> ```python
> if torch.cuda.is_available():
>     device = "cuda"
> elif torch.backends.mps.is_available():
>     device = "mps"
> else:
>     device = "cpu"
> ```

### Counting devices

Useful if you have access to more than one GPU — you can then run different processes on different GPUs.

In [45]:
# Count number of devices
import torch
torch.cuda.device_count()

0

### Putting tensors on the GPU

Call `.to(device)` on a tensor to move it. This returns a **copy** on the new device — reassign it to actually move it: `tensor = tensor.to(device)`.

In [46]:
# Create tensor (default on CPU)
tensor = torch.tensor([1, 2, 3])
print(tensor, tensor.device)

# Move tensor to GPU (if available)
tensor_on_gpu = tensor.to(device)
print(tensor_on_gpu)

tensor([1, 2, 3]) cpu
tensor([1, 2, 3])


### Moving tensors back to the CPU

NumPy can't work with GPU tensors directly, so if you need a tensor back in NumPy form, first bring it back to the CPU with `.cpu()`.

In [ ]:
# If a tensor is on the GPU, .numpy() will error - .cpu() first to copy it back to CPU memory
try:
    tensor_on_gpu.numpy()
except TypeError as e:
    print("TypeError:", e)

tensor_back_on_cpu = tensor_on_gpu.cpu().numpy()
tensor_back_on_cpu

array([1, 2, 3])